<a href="https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

## 1. Ranked actions + reason codes

The action queue ranks content by the model's estimated probability of decline, with higher-risk items appearing first. Actions are intended as decision-support rather than automatic instructions.

* **REFRESH** — highest-priority content with estimated decline risk ≥ 0.70.
* **REVIEW** — moderate-risk content with estimated decline risk from 0.45 to 0.70.
* **MONITOR** — lower-risk content with estimated decline risk below 0.45.

Reason codes translate the ranking into signals a human reviewer can understand. They use content staleness and recent impressions alongside the model score. The queue is a prioritization aid, not evidence that refreshing a page will cause better performance.


In [1]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load data
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "thahsinj06/Fly-rank-ml-internship-work/"
    "main/data/raw/content_refresh_anonymized.csv"
)
df = pd.read_csv(DATA_URL)

# Create target
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Week-5 leakage-safe feature set
features = [
    "search_volume",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr"
]

X = df[features]
y = df["is_declining_label"]

# Same model family used in Week 5, with median imputation
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X, y)

# Estimate decline probability for every content item
df["decline_probability"] = model.predict_proba(X)[:, 1]

# Action thresholds
df["action"] = np.select(
    [
        df["decline_probability"] >= 0.70,
        df["decline_probability"] >= 0.45
    ],
    [
        "REFRESH",
        "REVIEW"
    ],
    default="MONITOR"
)

# Human-readable reason codes
df["reason_code"] = np.select(
    [
        (df["days_since_last_update"] > 90) & (df["impressions_90d"] >= df["impressions_90d"].median()),
        (df["days_since_last_update"] > 90),
        (df["impressions_90d"] >= df["impressions_90d"].median()),
        (df["days_since_last_update"] <= 30)
    ],
    [
        "STALE_HIGH_VISIBILITY",
        "STALE_CONTENT",
        "HIGH_VISIBILITY",
        "RECENTLY_UPDATED"
    ],
    default="GENERAL_RISK"
)

# Rank highest-risk content first
queue = df.sort_values(
    "decline_probability",
    ascending=False
).reset_index(drop=True)

queue["priority_rank"] = np.arange(1, len(queue) + 1)

# Keep only useful fields for the action queue
queue = queue[
    [
        "priority_rank",
        "content_id",
        "action",
        "reason_code",
        "decline_probability",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr"
    ]
]

# Summary check
print("Queue size:", len(queue))
print("\nAction counts:")
print(queue["action"].value_counts())

print("\nTop 10 ranked actions:")
display(queue.head(10))

Queue size: 30000

Action counts:
action
REVIEW     29120
MONITOR      795
REFRESH       85
Name: count, dtype: int64

Top 10 ranked actions:


,priority_rank,content_id,action,reason_code,decline_probability,days_since_last_update,impressions_90d,avg_position,ctr
0,1,content_55a5b1c46474,REFRESH,STALE_CONTENT,0.836339,373,35,7.5,0.0
1,2,content_1b4ec72dafd4,REFRESH,STALE_CONTENT,0.836174,372,2,7.0,0.0
2,3,content_f6fdf87348f6,REFRESH,STALE_CONTENT,0.815974,373,2,32.5,0.0
3,4,content_06e19c6486b0,REFRESH,STALE_CONTENT,0.815014,334,10,5.0,0.0
4,5,content_8d56efff1e71,REFRESH,STALE_CONTENT,0.813200,372,1,35.0,0.0
5,6,content_e2b702f4f92b,REFRESH,STALE_CONTENT,0.811291,334,30,9.3,0.0
6,7,content_ccf25ed65a99,REFRESH,STALE_CONTENT,0.800685,305,2,0.0,0.0
7,8,content_02b0d6e30129,REFRESH,STALE_CONTENT,0.799359,313,176,6.9,0.0
8,9,content_f488400fca67,REFRESH,STALE_CONTENT,0.795375,305,155,5.7,0.0
9,10,content_7736e6144f3b,REFRESH,STALE_CONTENT,0.795245,304,11,5.2,0.0


## 2. Intended use and limits

## 2. Intended use and limits

The playbook is intended for content and SEO teams to **prioritize pages for human review**. It can help identify content that shows a higher measured likelihood of decline and organize review work by priority.

The model is **decision-support only**. It does not determine whether a page should actually be refreshed, rewritten, redirected, or removed.

The Week-5 model measured an F1 score of **0.681** on the random test split and **0.648** on the stricter client-grouped split. Recall remained high, but precision was more limited. This means the queue can be useful for screening, while some flagged pages will not actually be declining.

The results are directional and should not be treated as causal evidence or guaranteed future performance. The model should not be used outside the feature and data conditions represented in this analysis without further validation.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list


Every recommended action must be reviewed by a person before implementation.

The reviewer should check the page's current search intent, content quality, relevance, recent traffic trend, business importance, and whether the page has already been updated. The model score should be treated as a prioritization signal, not as the reason for the final decision.

The following actions should **never be automated from this playbook alone**:

* Automatically deleting or redirecting content
* Automatically changing search intent or targeting
* Automatically publishing rewritten content
* Automatically changing important business or product information
* Treating a high decline score as proof that a refresh will improve performance

The final action remains a human decision supported by the model and the available evidence.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers



We should check the model from time to time to make sure it is still useful.

We should review or retrain the model if:

* The data starts looking very different from the data used to train the model.
* The model's accuracy or F1 score becomes much lower.
* The model starts marking almost everything as REFRESH, REVIEW, or MONITOR.
* The way people search or the content strategy changes a lot.
* The data collection process changes.

These are warning signs, not automatic retraining rules. A new model should be tested before using it.


In [2]:
# Section 4: monitor the current recommendation distribution

action_distribution = (
    queue["action"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Current action distribution (%):")
print(action_distribution)

print("\nCurrent feature medians:")
print(df[features].median().round(3))

print("\nUse these values as a reference point for future monitoring.")

Current action distribution (%):
action
REVIEW     97.07
MONITOR     2.65
REFRESH     0.28
Name: proportion, dtype: float64

Current feature medians:
search_volume              10.00
impressions_90d           731.00
days_since_last_update     20.00
avg_position               10.80
ctr                         0.07
dtype: float64

Use these values as a reference point for future monitoring.


## 5. Exports for the paper



The final ranked queue is saved as a CSV file so it can be used in the report and paper.

The file contains the priority rank, recommended action, reason code, model score, and the main content signals used for review.


In [3]:
# Save the final action queue

import os

output_dir = "../../work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, "content_action_playbook.csv")

queue.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Rows:", len(queue))
print("Columns:", list(queue.columns))

Saved: ../../work/outputs/content_action_playbook.csv
Rows: 30000
Columns: ['priority_rank', 'content_id', 'action', 'reason_code', 'decline_probability', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.